# Phase 2 — Multi-Seed, Multi-Encoder Training Matrix

Chapter 4 currently reports a **single run** of a single encoder. A single run gives no
handle on seed variance, so it cannot support a claim that one encoder beats another. This
notebook replaces the point estimates with mean +/- sd over three seeds per configuration
and settles the encoder question on our own data.

**The encoder question.** A panelist's published work reports XLNet outperforming BERT on
privacy-policy classification; the thesis claims legal-domain pretraining dominates. Both
claims are about *other* corpora. Running all four encoders through an identical protocol
on identical splits is the only way to know which holds here.

## Run matrix

| Axis | Values |
| --- | --- |
| Encoders (dual-head) | `nlpaueb/legal-bert-base-uncased`, `bert-base-uncased`, `xlnet-base-cased`, `roberta-base` |
| Seeds | 42, 1337, 2024 |
| Head ablation (**all four encoders**) | topic-only, risk-only (dual-head reuses the runs above) |

**12 dual-head runs + 24 ablation runs = 36 (4 encoders x 3 head configs x 3 seeds).**

The ablation was originally run for legal-bert only, on the argument that "does joint
training help?" is a question about the architecture, not the backbone. That was an
assumption, not a measurement, and compute is no longer the binding constraint, so it is
now tested directly: the ablation is run for all four encoders and the dual-vs-single-head
delta is compared across them. XLNet is the encoder most likely to break the pattern — its
seed-to-seed sd is 5-10x every BERT-family encoder's and it pools its summary token from
the last position rather than the first — so an "architecture, not backbone" claim has to
survive XLNet to be worth making. The cross-encoder consistency check at the end of the
notebook is the deliverable; the extra 18 runs only exist to feed it.

## What is held fixed

Everything except encoder / seed / head mode, and it is held fixed by construction — the
runner imports `scripts/lawgic_train_matrix.py`, which reads the same persisted seed-42
split file, the same taxonomy, the same masked-BCE + masked-CE losses (copied line for
line from the original `DualHeadTrainer`), lr 3e-5, batch 8, up to 20 epochs, early
stopping patience 3, weight decay 0.01, warmup 0.06, FP16 on CUDA, max_length 256, and the
same pre-training degenerate-model assertion (a zero-logit model must score topic macro-F1
below 0.95).

## How the pooled representation is chosen per architecture

The two linear heads read one vector per clause. Which token that vector comes from is
**not** the same across these four encoders, and getting it wrong silently cripples a
model rather than erroring:

- **BERT, Legal-BERT, RoBERTa** — the sequence summary is the **first** token
  (`[CLS]` / `<s>`), placed there during pretraining.
- **XLNet** — XLNet is trained with the summary token **appended at the end**. Reading
  position 0 would hand the head an ordinary content token. So XLNet uses the **last**
  token.

`pooled_representation()` in `scripts/lawgic_train_matrix.py` is the single place this
lives. It selects by attention mask rather than by fixed index (`attention_mask.argmax(1)`
for first, `L - 1 - flip(mask).argmax(1)` for last), because XLNet's tokenizer pads on the
**left** while the BERT-family tokenizers pad on the right — a hardcoded `[:, 0]` or
`[:, -1]` would read padding for one of them.

Two further per-architecture quirks are handled in the same adapter, not scattered around:

- **RoBERTa has no `token_type_ids`.** The collator keeps only the keys in
  `tokenizer.model_input_names`, so each tokenizer declares its own contract and no
  `if roberta:` branch is needed anywhere.
- **XLNet's tokenizer needs `sentencepiece`.** Already present in
  `notebooks/requirements.txt` (`sentencepiece==0.2.1`); listed as a manual check below.

**Deviation to record in the manuscript.** The original v3 checkpoint fed the heads BERT's
`pooler_output` (a dense+tanh layer on top of `[CLS]`). The matrix uses the raw first
token instead, for all encoders. Reason: `roberta-base` ships with a *randomly initialised*
pooler, so keeping `pooler_output` would have handicapped RoBERTa for reasons unrelated to
the encoder itself. Consistency across the four arms matters more than bit-matching the
old run, so the legal-bert/seed-42 cell of this matrix is **not** expected to reproduce the
v3 checkpoint exactly — treat the matrix as internally comparable and the Phase 1 numbers
as the checkpoint's own.

In [1]:
import os
import sys
from pathlib import Path

# ── Corpus version: set BEFORE importing lawgic_eval_core ─────────────────────
os.environ["LAWGIC_CORPUS_VERSION"] = "v2"


def find_project_root(start: Path) -> Path:
    for sentinel in [
        "generated_files/lawgic_taxonomy/lawgic_multihead_wide_v2.csv",
        "generated_files/lawgic_taxonomy/lawgic_multihead_wide.csv",
    ]:
        for candidate in (start, *start.parents):
            if (candidate / sentinel).exists():
                return candidate
    raise FileNotFoundError("Run this notebook from inside the lawgic repository.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import json

import numpy as np
import pandas as pd

import lawgic_eval_core as core
import lawgic_train_matrix as tm

pd.set_option("display.width", 160)

split_path = core.persist_split()
corpus = core.load_corpus()
frames = core.split_frames(corpus)
assert {k: len(v) for k, v in frames.items()} == core.EXPECTED_SPLIT_ROWS
print(f"Split artifact: {split_path}")
print(f"Corpus version: {core._CORPUS_VERSION}")
print(f"Topics: {core.NUM_LAWGIC_TOPICS}")
print(f"Runs dir: {tm.RUNS_DIR}")
print("Rows:", {k: len(v) for k, v in frames.items()})

MATRIX = tm.build_matrix()
print(f"\nConfigured runs: {len(MATRIX)}")
display(pd.DataFrame([{
    "run_id": c.run_id, "encoder": c.encoder_name, "seed": c.seed,
    "heads": c.heads, "selection_metric": c.best_metric_key,
} for c in MATRIX]))


### Extending the matrix

`build_matrix()` returns the original 18 runs (4 encoders x 3 seeds dual-head, plus
legal-bert topic-only/risk-only x 3 seeds). `MATRIX` is a list of `RunConfig` dataclasses,
so extra arms are appended here rather than by rewriting `build_matrix()` — the function
stays the record of what Phase 2 originally ran.

The cell below appends the **18 missing ablation runs**: topic-only and risk-only, three
seeds each, for BERT, XLNet and RoBERTa. `RunConfig.best_metric_key` gives risk-only runs
`risk_macro_f1` and everything else `topic_macro_f1` automatically, so the selection
asymmetry that the legal-bert ablation already uses is replicated for the new encoders by
construction, not by hand. Nothing else about the protocol is touched.

The commented line keeps the earlier five-seed option available. Do **not** add seeds to
only some arms and then compare sds across arms — the sd of 5 draws is not comparable to
the sd of 3.

In [ ]:
MATRIX += [
    tm.RunConfig(encoder_name=encoder, seed=seed, heads=heads)
    for encoder in tm.ENCODERS[1:]
    for heads in ("topic", "risk")
    for seed in tm.SEEDS
]
assert len(MATRIX) == 36 and len({c.run_id for c in MATRIX}) == 36, len(MATRIX)
print(f"Runs after extension: {len(MATRIX)}")
display(pd.DataFrame([{
    "run_id": c.run_id, "encoder": c.encoder_name, "seed": c.seed,
    "heads": c.heads, "selection_metric": c.best_metric_key,
} for c in MATRIX if c.heads != "dual"]))

# MATRIX += [tm.RunConfig(encoder_name=tm.ENCODERS[0], seed=s, heads="dual") for s in (7, 2718)]


## MANUAL STEP — before running the matrix

1. **Model downloads.** The first run of each encoder pulls weights from the HuggingFace
   hub (~440 MB each for `bert-base-uncased`, `xlnet-base-cased`, `roberta-base`;
   legal-bert is already local). Requires network access on the training machine. The
   cell below pre-fetches all three in-notebook via `AutoModel`/`AutoTokenizer`.
2. **`sentencepiece`** must be importable for the XLNet tokenizer. It is already in
   `notebooks/requirements.txt`; the check cell below verifies it rather than installing it.
3. **GPU.** These are 36 full fine-tunes. On CPU this is days, not hours — run on the CUDA
   machine that produced the v3 checkpoint. FP16 switches on automatically on CUDA and off
   elsewhere, matching the original protocol.
4. **Disk.** Each run keeps one checkpoint (`save_total_limit=1`), ~440 MB, plus a small
   `test_logits.npz`. Budget ~20 GB for the full matrix under
   `generated_files/lawgic_taxonomy/runs/`.

Nothing here writes to `saved_models/`; the deployed v3 checkpoint is never touched.

In [ ]:
from transformers import AutoModel, AutoTokenizer

MODELS_TO_PREFETCH = ["bert-base-uncased", "xlnet-base-cased", "roberta-base"]

for model_name in MODELS_TO_PREFETCH:
    print(f"Downloading {model_name} ...")
    AutoTokenizer.from_pretrained(model_name)
    AutoModel.from_pretrained(model_name)
    print(f"  done: {model_name}")

print("\nAll three encoders cached locally.")

In [ ]:
%conda install conda-forge::sentencepiece

In [ ]:
import importlib.util

print("sentencepiece:", "OK" if importlib.util.find_spec("sentencepiece") else "MISSING — XLNet will fail")
print("scipy:", "OK" if importlib.util.find_spec("scipy") else "MISSING — McNemar will fail")

device_label, device = tm.detect_device()
print(f"device: {device_label} (fp16={device_label == 'cuda'})")
if device_label != "cuda":
    print("WARNING: not on CUDA. The matrix will take days. Stop and move to the GPU machine.")

## Expected wall time

The training checkpoint's `trainer_state.json` records epochs before early stopping
at batch size 8. The cell below derives a lower-bound estimate from the eval throughput
recorded in the most recent checkpoint's trainer state, if available.


In [ ]:
state_path = core.CHECKPOINT_DIR / "checkpoints/checkpoint-45016/trainer_state.json"
if state_path.exists():
    state = json.loads(state_path.read_text())
    evals = [h for h in state["log_history"] if "eval_runtime" in h]
    eval_throughput = float(np.mean([h["eval_samples_per_second"] for h in evals]))
    epochs = float(state["epoch"])
    # Training is roughly 3-4x the cost of inference per sample (forward + backward + optimizer).
    optimistic_seconds = epochs * (len(frames["train"]) / (eval_throughput / 3.5))
    print(f"v3 run: {epochs:.0f} epochs, eval throughput {eval_throughput:.0f} clauses/s")
    print(f"Derived LOWER BOUND per run: ~{optimistic_seconds / 60:.0f} min "
          f"-> ~{len(MATRIX) * optimistic_seconds / 3600:.1f} h for {len(MATRIX)} runs")
    print("This is an extrapolation, not a measurement. Trust wall_seconds from run 1 instead.")
else:
    print("No v3 trainer_state.json found; wall time must be measured on the first run.")

## Runner

Each config trains, evaluates on the frozen test split, and writes to
`generated_files/lawgic_taxonomy/runs/<run_id>/`:

- `metrics.json` — config + headline test metrics + `wall_seconds` + `epochs_run`
- `test_logits.npz` — test logits, labels and masks (so aggregation, bootstrap and paired
  tests never need to re-run inference)
- `per_topic.csv` — per-topic precision / recall / F1 / support

Completed runs are skipped, so the cell is **resumable**: interrupt it, restart the kernel,
re-run. Set `FORCE_RERUN = True` to redo everything.

### Best-model export helper

Defined before the runner so each encoder's best checkpoint is written to `saved_models/` as soon as its runs finish, rather than only after all 36.

In [5]:
# Best-model export, defined BEFORE the runner so the matrix can save each
# encoder's best checkpoint to saved_models/ as soon as that encoder's runs
# finish, instead of only after all 36 runs complete. Interrupting the matrix
# therefore never loses an already-trained model.
#
# Layout matches lawgic_classifier_legal-bert_v3 (model_state_dict.pt +
# encoder/tokenizer + head weights + taxonomy + metadata). Reads metrics.json
# directly from disk, so it is resumable across kernel restarts. Writes to a NEW
# directory per encoder (suffix "_phase2") - never touches
# lawgic_classifier_legal-bert_v3.
#
# Called after every dual-head run, so the export is redone when a later seed
# beats the currently exported one; if the exported directory already holds the
# best run it is left untouched.

import shutil
from datetime import datetime, timezone

import torch
from safetensors.torch import load_file as load_safetensors
from transformers import AutoTokenizer

SAVE_TARGETS = {
    "nlpaueb/legal-bert-base-uncased": "legal-bert",
    "bert-base-uncased": "bert",
    "xlnet-base-cased": "xlnet",
    "roberta-base": "roberta",
}
SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models"


def completed_dual_runs(encoder_name: str) -> list[dict]:
    records = []
    for metrics_path in sorted(tm.RUNS_DIR.glob("*/metrics.json")):
        record = json.loads(metrics_path.read_text())
        if record["encoder_name"] == encoder_name and record["heads"] == "dual":
            records.append(record)
    return records


def best_checkpoint_dir(run_id: str) -> Path:
    checkpoints = sorted(
        (tm.RUNS_DIR / run_id / "checkpoints").glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]),
    )
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoint saved for {run_id}")
    # save_total_limit=1 + load_best_model_at_end=True: the one surviving
    # checkpoint is the best validation checkpoint, not just the last epoch.
    return checkpoints[-1]


def save_best_model(encoder_name: str, short_name: str | None = None) -> None:
    short_name = short_name or SAVE_TARGETS[encoder_name]
    candidates = completed_dual_runs(encoder_name)
    if not candidates:
        print(f"skip {short_name}: no completed dual-head runs yet")
        return

    best = max(candidates, key=lambda r: r["best_val_metric"])
    run_id = best["run_id"]

    output_dir = SAVED_MODELS_DIR / f"lawgic_classifier_{short_name}_phase2"
    if output_dir.exists():
        existing = output_dir / "training_metadata.json"
        exported_run = (
            json.loads(existing.read_text())["source_run_id"] if existing.exists() else None
        )
        if exported_run == run_id:
            print(f"skip {short_name}: {run_id} already exported")
            return
        print(f"[{short_name}] {exported_run} superseded by {run_id}, re-exporting")
        shutil.rmtree(output_dir)

    checkpoint_dir = best_checkpoint_dir(run_id)

    model = tm.LawgicDualHeadModel(encoder_name)
    weights_file = checkpoint_dir / "model.safetensors"
    state_dict = (
        load_safetensors(str(weights_file))
        if weights_file.exists()
        else torch.load(checkpoint_dir / "pytorch_model.bin", map_location="cpu", weights_only=True)
    )
    model.load_state_dict(state_dict)

    tokenizer = AutoTokenizer.from_pretrained(str(checkpoint_dir))

    output_dir.mkdir(parents=True)

    # Full state dict + encoder/tokenizer + heads separately, mirroring v3's layout.
    torch.save(model.state_dict(), output_dir / "model_state_dict.pt")
    model.encoder.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    torch.save(model.topic_head.state_dict(), output_dir / "topic_head_weights.pt")
    torch.save(model.harm_head.state_dict(), output_dir / "harm_head_weights.pt")

    topic_ids, name_by_topic, _ = core.load_taxonomy()
    compact_taxonomy = [
        {"classifier_id": i, "topic_id": tid, "name": name_by_topic[tid]}
        for i, tid in enumerate(topic_ids)
    ]
    (output_dir / f"lawgic_topics_{core.NUM_LAWGIC_TOPICS}.json").write_text(json.dumps(compact_taxonomy, indent=2))
    shutil.copy2(core.TAXONOMY_PATH, output_dir / f"lawgic_topics_original_{core.TOTAL_TAXONOMY_TOPICS}.json")

    (output_dir / "test_metrics.json").write_text(json.dumps(best, indent=2, default=str))

    metadata = {
        "model_name": encoder_name,
        "architecture": "dual_head",
        "num_topics": core.NUM_LAWGIC_TOPICS,
        "num_harm_classes": core.NUM_HARM_CLASSES,
        "max_length": core.MAX_LENGTH,
        "decision_threshold": core.DECISION_THRESHOLD,
        "seed": best["seed"],
        "source_run_id": run_id,
        "best_val_metric": best["best_val_metric"],
        "seeds_considered": sorted(r["seed"] for r in candidates),
        "saved_at": datetime.now(timezone.utc).isoformat(),
        "note": (
            "Best-of-3-seeds model from the Phase 2 multi-encoder matrix "
            "(notebooks/evaluation/02_multiseed_encoder_runs.ipynb); does not "
            "replace the primary checkpoint."
        ),
    }
    (output_dir / "training_metadata.json").write_text(json.dumps(metadata, indent=2))

    print(f"[{short_name}] saved best seed {best['seed']} (run {run_id}) -> {output_dir}")



In [ ]:
FORCE_RERUN = False

records = []
for index, config in enumerate(MATRIX, start=1):
    target = tm.RUNS_DIR / config.run_id / "metrics.json"
    if target.exists() and not FORCE_RERUN:
        print(f"[{index}/{len(MATRIX)}] skip {config.run_id} (already complete)")
        records.append(json.loads(target.read_text()))
    else:
        print(f"[{index}/{len(MATRIX)}] running {config.run_id} ...")
        records.append(tm.run_config(config))
    # Export this encoder's best dual-head run as soon as it is known, so an
    # interrupted matrix keeps every model it has already trained. Runs on the
    # skip branch too: the dual arms may already be on disk from an earlier
    # session, and the export is idempotent (it compares source_run_id).
    if config.heads == "dual":
        save_best_model(config.encoder_name)

print(f"Completed {len(records)} runs.")

Re-export the best models (no-op if the runner already did)

In [ ]:
# Safety net: after the full matrix, confirm every encoder's best dual-head run
# is the one exported. A no-op when the in-loop export already did it, and the
# way models get written if the runner cell was skipped entirely.
for encoder_name, short_name in SAVE_TARGETS.items():
    save_best_model(encoder_name, short_name)


## Aggregation

Everything below reads the persisted run artifacts, so it can be re-run without a GPU.

In [ ]:
run_files = sorted(tm.RUNS_DIR.glob("*/metrics.json"))
runs = pd.DataFrame([json.loads(p.read_text()) for p in run_files])
runs = runs[runs["holdout_source"].isna()] if "holdout_source" in runs else runs
print(f"Loaded {len(runs)} Phase 2 runs from {tm.RUNS_DIR}")
display(runs[["run_id", "encoder_name", "seed", "heads", "epochs_run", "wall_seconds", *core.HEADLINE_METRICS]])

runs.to_csv(core.EVAL_OUT_DIR / "phase2_runs.csv", index=False)
print(f"\nMeasured wall time: {runs['wall_seconds'].mean() / 60:.1f} min/run "
      f"(total {runs['wall_seconds'].sum() / 3600:.1f} h)")

In [ ]:
grouped = runs.groupby(["encoder_name", "heads"])
aggregate = grouped[list(core.HEADLINE_METRICS)].agg(["mean", "std", "count"])
aggregate.columns = ["_".join(c) for c in aggregate.columns]
aggregate = aggregate.reset_index()
display(aggregate)
aggregate.to_csv(core.EVAL_OUT_DIR / "phase2_aggregate.csv", index=False)

### Bootstrap CIs on the test metrics

Per run, 1,000 clause-level resamples of the test split, computed from the stored logits.
Reported alongside the across-seed sd: the bootstrap CI measures *test-set* sampling
noise, the sd measures *initialisation/ordering* noise. They are different quantities and
the manuscript should not conflate them.

In [ ]:
N_RESAMPLES = 1000


def load_run_logits(run_id: str) -> dict:
    payload = np.load(tm.RUNS_DIR / run_id / "test_logits.npz")
    return {
        "topic_logits": payload["topic_logits"],
        "harm_logits": payload["harm_logits"],
        "arrays": {
            "labels": payload["labels"],
            "label_masks": payload["label_masks"],
            "harm_labels": payload["harm_labels"],
            "harm_masks": payload["harm_masks"],
        },
        "row_id": payload["row_id"],
    }


ci_rows = []
for run_id in runs["run_id"]:
    payload = load_run_logits(run_id)
    ci = core.bootstrap_ci(
        payload["topic_logits"], payload["harm_logits"], payload["arrays"], n_resamples=N_RESAMPLES
    )
    ci.insert(0, "run_id", run_id)
    ci_rows.append(ci)

bootstrap_table = pd.concat(ci_rows, ignore_index=True)
bootstrap_table.to_csv(core.EVAL_OUT_DIR / "phase2_bootstrap_ci.csv", index=False)
display(bootstrap_table.head(12))

### Paired significance tests

Both tests are **paired on the clause**: every run scored the identical test rows in the
identical order, so a difference is attributable to the varied component and nothing else.

- **Risk head — McNemar.** Item-level correctness per clause (over `harm_mask=1` rows),
  exact binomial on the discordant pairs. This is the right test for two classifiers on
  one sample; an unpaired accuracy comparison would throw away the pairing and lose power.
- **Topic head — paired bootstrap.** Macro-F1 is not decomposable into per-item
  correctness, so McNemar does not apply. Instead each resample draws one set of clause
  indices and scores *both* models on it; the reported interval is over the difference.

Seeds are averaged out by comparing the **best seed** of each arm; change `pick` below to
compare a fixed seed if you would rather not condition on validation performance.

In [ ]:
def best_run(encoder: str, heads: str = "dual") -> str:
    subset = runs[(runs["encoder_name"] == encoder) & (runs["heads"] == heads)]
    if subset.empty:
        raise KeyError(f"no runs for {encoder}/{heads}")
    return subset.sort_values("best_val_metric", ascending=False).iloc[0]["run_id"]


def compare(run_a: str, run_b: str) -> dict:
    a, b = load_run_logits(run_a), load_run_logits(run_b)
    assert np.array_equal(a["row_id"], b["row_id"]), "runs were scored on different rows"
    arrays = a["arrays"]

    valid = arrays["harm_masks"].astype(bool)
    correct_a = a["harm_logits"][valid].argmax(1) == arrays["harm_labels"][valid]
    correct_b = b["harm_logits"][valid].argmax(1) == arrays["harm_labels"][valid]
    mcnemar = core.mcnemar(correct_a, correct_b)

    def delta(indices):
        ma = core.topic_metrics(a["topic_logits"][indices], arrays["labels"][indices], arrays["label_masks"][indices])
        mb = core.topic_metrics(b["topic_logits"][indices], arrays["labels"][indices], arrays["label_masks"][indices])
        return ma["topic_macro_f1"] - mb["topic_macro_f1"]

    paired = core.paired_bootstrap_delta(delta, np.arange(len(arrays["labels"])), n_resamples=N_RESAMPLES)
    return {
        "run_a": run_a,
        "run_b": run_b,
        "risk_mcnemar_b": mcnemar["b"],
        "risk_mcnemar_c": mcnemar["c"],
        "risk_mcnemar_p": mcnemar["p_value"],
        "topic_macro_f1_delta": paired["delta"],
        "topic_delta_ci_low": paired["ci_low"],
        "topic_delta_ci_high": paired["ci_high"],
        "topic_delta_p": paired["p_value"],
    }


LEGAL_BERT = tm.ENCODERS[0]
comparisons = []
for other in tm.ENCODERS[1:]:
    try:
        comparisons.append(compare(best_run(LEGAL_BERT), best_run(other)))
    except KeyError as exc:
        print(f"skipped: {exc}")

# Head ablation, original scope: dual vs each single-head variant, legal-bert only.
# Kept verbatim so the Section 4.4.3 numbers stay traceable to the cell that produced them.
for heads in ("topic", "risk"):
    try:
        comparisons.append(compare(best_run(LEGAL_BERT), best_run(LEGAL_BERT, heads)))
    except KeyError as exc:
        print(f"skipped: {exc}")

# Head ablation, extended scope: the same two comparisons for the other three encoders,
# same bootstrap / McNemar apparatus, no new test. Legal-BERT is skipped here because the
# loop above already appended it.
ablation_rows = []
for encoder in tm.ENCODERS:
    for heads in ("topic", "risk"):
        try:
            row = compare(best_run(encoder, "dual"), best_run(encoder, heads))
        except KeyError as exc:
            print(f"skipped: {exc}")
            continue
        ablation_rows.append({"encoder_name": encoder, "heads": heads, **row})
        if encoder != LEGAL_BERT:
            comparisons.append(row)

ablation = pd.DataFrame(ablation_rows)
ablation.to_csv(core.EVAL_OUT_DIR / "phase2_head_ablation.csv", index=False)
display(ablation)

significance = pd.DataFrame(comparisons)
significance.to_csv(core.EVAL_OUT_DIR / "phase2_significance.csv", index=False)
display(significance)

## Output table (a) — headline, rows = encoder/config

Cells are `mean +/- sd` over seeds. `n/a` marks a metric a configuration cannot produce:
topic-only leaves the risk head untrained, risk-only leaves the topic head untrained, so
reporting those cells would be reporting random weights.

In [ ]:
LABELS = {
    "topic_macro_f1": "Topic macro-F1",
    "topic_micro_f1": "Topic micro-F1",
    "risk_accuracy": "Risk accuracy",
    "risk_macro_f1": "Risk macro-F1",
}
SHORT_NAMES = {
    "nlpaueb/legal-bert-base-uncased": "Legal-BERT",
    "bert-base-uncased": "BERT",
    "xlnet-base-cased": "XLNet",
    "roberta-base": "RoBERTa",
}
HEAD_NAMES = {"dual": "dual", "topic": "topic-only", "risk": "risk-only"}
# 4 encoders x 3 head configs, encoder-major so each encoder's three rows sit together.
CONFIG_NAMES = {
    (encoder, heads): f"{SHORT_NAMES[encoder]} ({HEAD_NAMES[heads]})"
    for encoder in tm.ENCODERS
    for heads in ("dual", "topic", "risk")
}


def mean_sd(values: pd.Series) -> str:
    if values.isna().all():
        return "n/a"
    return f"{values.mean():.3f} ± {values.std(ddof=1):.3f}" if len(values) > 1 else f"{values.mean():.3f}"


headline = pd.DataFrame(
    [
        {
            "Configuration": CONFIG_NAMES.get((encoder, heads), f"{encoder} ({heads})"),
            "Seeds": int(group["seed"].nunique()),
            **{LABELS[m]: mean_sd(group[m]) for m in core.HEADLINE_METRICS},
        }
        for (encoder, heads), group in runs.groupby(["encoder_name", "heads"])
    ]
)
order = [CONFIG_NAMES[k] for k in CONFIG_NAMES if CONFIG_NAMES[k] in set(headline["Configuration"])]
headline = headline.set_index("Configuration").loc[order].reset_index()
display(headline)

core.write_outputs(
    headline,
    "phase2_headline",
    caption=(
        "Test performance by encoder and head configuration, mean $\\pm$ standard deviation "
        "over three seeds (42, 1337, 2024), for the full 4 encoder x 3 head-config design. "
        "All runs use the identical persisted seed-42 "
        "clause split and identical hyperparameters; only the encoder, the seed and the "
        "active heads vary."
    ),
    label="tab:encoder-matrix",
)

## Cross-encoder consistency of the dual-head benefit

The point of running the ablation on all four encoders is not four more rows of numbers —
it is whether the four deltas *agree*. Two views of the same quantity are reported side by
side, and neither is collapsed into a grand mean:

- **Seed-level delta.** Per encoder, dual-head minus single-head on the metric that
  single-head arm still produces (risk macro-F1 for dual-vs-risk-only, topic macro-F1 for
  dual-vs-topic-only), paired by seed, reported as mean +/- sd over the three seed pairs.
  With n = 3 the interval shown is mean +/- t(0.975, 2) * sd / sqrt(3) — wide by
  construction, and that width is the honest statement of what three seeds buy.
- **Best-seed delta.** The McNemar b/c/p (risk) and paired-bootstrap CI (topic) already
  computed in the ablation cell above, on the best-validation seed of each arm. Same
  machinery as every other comparison in this notebook and in the chapter.

The check reported at the end: do all four seed-level intervals overlap a common value,
and does any single encoder's interval sit clear of the other three's range. Overlap
supports "the dual-head benefit is a property of the training objective"; a non-overlapping
encoder rejects it and the claim must be scoped to the backbones where it holds.

In [ ]:
# Metric each single-head arm can still be compared on. topic-only leaves the risk head
# untrained and risk-only leaves the topic head untrained, so each contrast has exactly one
# valid metric — the same "n/a" logic as the headline table.
ABLATION_METRIC = {"risk": "risk_macro_f1", "topic": "topic_macro_f1"}
T_CRIT = 4.303  # t(0.975, df=2): three seeds, two-sided 95%


def seed_level_delta(encoder: str, heads: str, metric: str) -> dict:
    """dual minus single-head on `metric`, paired seed by seed."""
    subset = runs[runs["encoder_name"] == encoder]
    dual = subset[subset["heads"] == "dual"].set_index("seed")[metric]
    single = subset[subset["heads"] == heads].set_index("seed")[metric]
    seeds = sorted(set(dual.index) & set(single.index))
    if not seeds:
        raise KeyError(f"no paired seeds for {encoder} dual vs {heads}")
    deltas = np.array([dual[s] - single[s] for s in seeds], dtype=float)
    half_width = T_CRIT * deltas.std(ddof=1) / np.sqrt(len(deltas)) if len(deltas) > 1 else np.nan
    return {
        "encoder": SHORT_NAMES[encoder],
        "contrast": f"dual - {HEAD_NAMES[heads]}",
        "metric": metric,
        "n_seeds": len(deltas),
        "delta_mean": deltas.mean(),
        "delta_sd": deltas.std(ddof=1) if len(deltas) > 1 else np.nan,
        "ci_low": deltas.mean() - half_width,
        "ci_high": deltas.mean() + half_width,
        "all_seeds_positive": bool((deltas > 0).all()),
        "per_seed": np.round(deltas, 4).tolist(),
    }


def overlap_report(table: pd.DataFrame, title: str) -> None:
    """Flag whether the per-encoder intervals share a common value, and name any outlier."""
    print(f"\n{title}")
    for _, row in table.iterrows():
        print(f"  {row['encoder']:<11} {row['delta_mean']:+.4f} +/- {row['delta_sd']:.4f} "
              f"[{row['ci_low']:+.4f}, {row['ci_high']:+.4f}]  seeds={row['per_seed']}")

    lower, upper = table["ci_low"].max(), table["ci_high"].min()
    if lower <= upper:
        print(f"  -> all {len(table)} intervals overlap on [{lower:+.4f}, {upper:+.4f}]: "
              "consistent with one common effect.")
    else:
        # No common value. Name the encoders whose interval is disjoint from every other's.
        outliers = [
            row["encoder"] for _, row in table.iterrows()
            if all(row["ci_high"] < o["ci_low"] or row["ci_low"] > o["ci_high"]
                   for _, o in table.iterrows() if o["encoder"] != row["encoder"])
        ]
        print("  -> NO common value: the four intervals do not share a point.")
        print(f"     disjoint from all others: {outliers or 'none individually — pairwise only'}")

    signs = set(np.sign(table["delta_mean"]))
    print(f"     sign agreement: {'all same sign' if len(signs) == 1 else 'SIGNS DISAGREE'}"
          f" ({', '.join(f'{r.encoder} {r.delta_mean:+.4f}' for r in table.itertuples())})")


consistency_rows = []
for heads, metric in ABLATION_METRIC.items():
    for encoder in tm.ENCODERS:
        try:
            consistency_rows.append(seed_level_delta(encoder, heads, metric))
        except KeyError as exc:
            print(f"skipped: {exc}")

consistency = pd.DataFrame(consistency_rows)

# Attach the best-seed significance already computed above, so the seed-level spread and
# the paired test sit in one table instead of two.
if not ablation.empty:
    keyed = ablation.set_index([ablation["encoder_name"].map(SHORT_NAMES),
                                ablation["heads"].map(lambda h: f"dual - {HEAD_NAMES[h]}")])
    for column in ("risk_mcnemar_b", "risk_mcnemar_c", "risk_mcnemar_p",
                   "topic_macro_f1_delta", "topic_delta_ci_low", "topic_delta_ci_high",
                   "topic_delta_p"):
        consistency[column] = [
            keyed[column].get((row.encoder, row.contrast), np.nan) for row in consistency.itertuples()
        ]
    # Blank the columns that do not apply to a contrast: McNemar is a risk-head test, the
    # paired bootstrap a topic-head one.
    risk_rows = consistency["metric"] == "risk_macro_f1"
    consistency.loc[risk_rows, ["topic_macro_f1_delta", "topic_delta_ci_low",
                                "topic_delta_ci_high", "topic_delta_p"]] = np.nan
    consistency.loc[~risk_rows, ["risk_mcnemar_b", "risk_mcnemar_c", "risk_mcnemar_p"]] = np.nan

consistency.to_csv(core.EVAL_OUT_DIR / "phase2_head_ablation_consistency.csv", index=False)
display(consistency)

risk_side = consistency[consistency["metric"] == "risk_macro_f1"].reset_index(drop=True)
topic_side = consistency[consistency["metric"] == "topic_macro_f1"].reset_index(drop=True)
if len(risk_side) > 1:
    overlap_report(risk_side, "Risk macro-F1: dual-head minus risk-only, per encoder")
if len(topic_side) > 1:
    overlap_report(topic_side, "Topic macro-F1: dual-head minus topic-only, per encoder")


## Output table (b) — per-topic breakdown for the final model

Rows = topics + macro avg + weighted avg, columns = precision / recall / F1 / support.
Reported twice: for the **best legal-bert seed** (the deployment candidate) and as the
**seed-mean F1** (how stable the per-topic ranking is across seeds).


In [ ]:
topic_ids, name_by_topic, _ = core.load_taxonomy()

best_legal_bert = best_run(LEGAL_BERT)
best_table = pd.read_csv(tm.RUNS_DIR / best_legal_bert / "per_topic.csv")

seed_tables = [
    pd.read_csv(tm.RUNS_DIR / run_id / "per_topic.csv").set_index("topic_id")
    for run_id in runs[(runs["encoder_name"] == LEGAL_BERT) & (runs["heads"] == "dual")]["run_id"]
]
mean_table = sum(t[["precision", "recall", "f1"]] for t in seed_tables) / len(seed_tables)
mean_table = mean_table.join(seed_tables[0][["support", "observed"]]).reset_index()

per_topic = best_table.merge(mean_table, on="topic_id", suffixes=("_best", "_mean"))
per_topic.insert(1, "topic_name", per_topic["topic_id"].map(lambda t: name_by_topic.get(t, t)))
display(per_topic)

core.write_outputs(
    per_topic[["topic_id", "topic_name", "precision_best", "recall_best", "f1_best",
               "f1_mean", "support_best"]].rename(columns={
        "topic_id": "Topic", "topic_name": "Name", "precision_best": "P", "recall_best": "R",
        "f1_best": "F1", "f1_mean": "F1 (seed mean)", "support_best": "Support"}),
    "phase2_per_topic",
    caption=(
        f"Per-topic test performance of the best Legal-BERT dual-head seed ({best_legal_bert}), "
        "with the mean F1 across the three seeds for comparison. Support counts supervised "
        "positive cells in the test split; topics with no observed test cells are omitted."
    ),
    label="tab:per-topic",
)